In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [0]:
print("=== [STREAMING MONITOR] UC Volume 实时流监听雷达开火 ===\n")

In [0]:
# 1. 强行固化 Schema 防火墙
order_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# 2. 启动实时流读取
df_stream = spark.readStream \
    .schema(order_schema) \
    .parquet("/Volumes/dbacademy/streaming/vol_yuto_stream_source/")

### 一、`df_stream` 到底是什么？
`df_stream` 不是普通静态 DataFrame（你前面 `spark.read.parquet` 读全量得到的是**批 DataFrame**），它是 **Spark Structured Streaming 流式无界 DataFrame**，官方叫：
**Streaming DataFrame（流式数据集）**

核心区别一句话：
- `spark.read` → 读**已经存在、固定不变**的一批文件，数据有终点，读完就结束（有界数据集 Batch DataFrame）
- `spark.readStream` → 持续监听目录，**等待未来不断新增的 parquet 文件**，数据没有终点（无界流 Streaming DataFrame）

#### 关键特性
1. **不立刻加载数据**
执行这行代码时不会去扫描文件、不会count、不会读内容，只是定义一套「读取规则」：schema、数据源目录、文件格式。
2. **惰性执行**
只有你调用 `.writeStream.start()` 启动流查询后，Spark 才会后台持续监控目录，发现新文件自动加载处理。
3. **代表无限数据流**
只要脚本不停止、流不关闭，后续你前面生成器每2秒写入一批新parquet，它都会自动捕获，实现实时增量读取。

### 二、每一段代码拆解：为什么要这么写
```python
df_stream = spark.readStream \
    .schema(order_schema) \
    .parquet("/Volumes/dbacademy/streaming/vol_yuto_stream_source/")
```
#### 1. `spark.readStream`
入口API，专门用来创建**流式读取器**，对应批处理的 `spark.read`。
- 批：`read` 一次性读完现有所有数据，任务跑完退出
- 流：`readStream` 开启长期运行的监听进程，持续等待新数据产生

#### 2. `.schema(order_schema)` 必须写，不能省略（重点）
### 批读取可以不强制指定schema：
`spark.read.parquet(path)` 会自动解析parquet文件内自带的元数据schema。
### 流式读取**强制手动提供schema**，原因：
流是无界的，启动时目录里可能**还没有任何文件**（还没生成模拟订单数据）。
没有文件 → Spark无法自动推断表结构，直接报错。
你提前定义好 `order_schema`，Spark不用依赖已有文件，直接知道：
字段名、字段类型、是否允许null，保证后续新增文件结构统一。

生产场景价值：防止上游写入脏文件、字段类型错乱导致流任务崩溃。

#### 3. `.parquet("目录路径")`
指定**流式数据源目录**，也就是你数据生成器不断输出parquet批次的根目录。
Spark Streaming对文件源的底层机制：
1. 启动时扫描目录下所有已存在的parquet文件，一次性加载历史存量数据；
2. 后台定时轮询该目录，检测是否新增文件/子文件夹；
3. 只要有新写入完成的parquet文件，自动作为新批次加载进流计算；
> 注意：文件写入必须完整关闭后才会被读取，如果写入中途，Spark不会读取半完成文件。


In [0]:
# 3. 流式解析与高危异常过滤Transformation
df_alerts = df_stream.filter(
    (F.col("amount") > 10000.0) | (F.col("status") == "FAILED")
).withColumn("audit_timestamp", F.current_timestamp())


#### 1. 什么叫「只是定义逻辑、不存数据、惰性执行」

##### 批处理（普通df，有界）
```python
df_audit = spark.read.parquet(path)
df_filter = df_audit.filter(条件)
df_filter.show()
```
执行到 `filter` 这一行，Spark**立刻扫描磁盘、加载数据、过滤**，内存里实实在在存在过滤后的结果，你马上能show、count看数据。

##### 流式 df_stream / df_alerts（无界）
```python
df_stream = spark.readStream.parquet(path)
df_alerts = df_stream.filter(...).withColumn(...)
```
写到这里，**Spark完全不读磁盘、不处理任何订单、内存里没有一条数据**。
这两行只是一张「执行计划书」：
> 以后只要有新订单文件流入，先全部读进来 → 过滤大额/失败单 → 新增审计时间戳

没有 `writeStream.start()` 启动任务，这份计划书永远不会运行，`df_alerts` 只是一套规则，没有真实数据。

#### 2. 为什么不能 .show() / .count()
普通 `.show()` / `.count()` 是同步批动作：要求数据集有头有尾、全部加载完毕再计算行数、打印结果。
但订单流是**无限持续产生**的，没有终点，Spark没法“读完所有数据再统计”，所以直接调用会抛错。
想看数据必须用流式专属输出 `writeStream`（控制台、表、文件等）持续输出增量。

### 二、df_alerts 告警流存在的核心意义
#### 意义1：分流，把业务两类数据彻底拆开，各司其职
原始流 `df_stream` = **全量订单大流**，包含正常单+风险异常单，混杂在一起。
绝大多数订单是正常SUCCESS小额单，我们根本不需要处理、存储、告警它们；
- 如果直接对全量流做存储/推送告警：浪费磁盘、网络、计算资源；
- `df_alerts` 只保留我们关心的高危异常数据，做数据裁剪，**只过滤出需要后续处理的数据**。

数据流分层：
1. 底层源流 df_stream：原始所有订单（原材料）
2. 中间转换流 df_alerts：筛选后的风险订单（精加工后的半成品，专门给告警系统使用）

#### 意义2：统一附加业务字段，标准化告警数据结构
你加了一列 `audit_timestamp`，这是告警系统必需字段：
- 原始数据只有下单业务时间 `event_time`；
- `audit_timestamp` 是系统识别到风险的时间，用来衡量告警延迟、排查流式任务卡顿。

所有下游告警消费方（日志表、钉钉/短信推送、监控大盘）统一拿到带审计时间的标准结构，不用每个下游重复写 `withColumn`，逻辑复用。

#### 意义3：解耦，计算逻辑和输出逻辑分离
##### 写法拆分
1. 转换层（只定义规则，纯计算）
```python
# 只关心：哪些是异常、补充什么字段
df_alerts = df_stream.filter(风险条件).withColumn("audit_timestamp", F.current_timestamp())
```
2. 输出层（只关心：告警发到哪里）
```python
# 分支1：异常打印控制台调试
df_alerts.writeStream.format("console").start()
# 分支2：异常存入告警数据表持久化
df_alerts.writeStream.format("delta").table("order_risk_alerts").start()
# 分支3：对接推送服务实时发短信告警
df_alerts.writeStream.foreach(PushAlert()).start()
```

##### 优势
- 过滤、打时间戳这套告警规则只写一次；
- 可以同时多目的地输出，不用重复写过滤逻辑；
- 后续修改风险阈值（比如金额从1万改成5万），只改df_alerts一行代码，所有下游自动生效。

#### 意义4：适配实时监控场景，只增量输出风险事件
生产实时监控核心诉求：**有异常才通知，正常订单静默忽略**。
`df_alerts` 天然只增量输出命中规则的新异常，搭配 `outputMode("append")`，每出现一笔高危订单就下发一条告警，完美匹配实时风控、实时监控的业务需求。

### 三、一个完整业务场景-直观理解
电商实时风控场景：
1. 模拟生成器不断产生订单存入文件夹；
2. `df_stream` 实时读取全部订单；
3. `df_alerts` 过滤出大额、失败订单，打上系统检测时间；
4. 把 df_alerts 输出到三处：
   1）控制台：开发调试看实时异常；
   2）风控告警Delta表：永久留存所有风险订单，用于事后对账复盘；
   3）自定义推送接口：一旦产生df_alerts数据，立刻推送钉钉告警给运营。

如果没有df_alerts，只能把全量订单全部下发，正常订单会疯狂刷屏、存储大量无用数据，推送接口不断接收无效消息，资源完全浪费。

### 四、一句话总结 df_alerts 的定位
`df_stream` 是完整、未加工的原始无限订单流；
`df_alerts` 是**经过业务规则筛选、标准化加工后的风险事件专用子流**；
存在价值：过滤无用数据、统一加工字段、复用计算逻辑，专门供给下游告警、风控、监控模块消费。

In [0]:
# 释放微批次引擎：每2秒切一刀，实时向控制台对账吐数
# 注意：额外挂载了一个内存缓存表，方便随时用 SQL 检查流

print("🚀 === [PRODUCTION SERVERLESS STREAMING] 启动挂载物理账本的弹性流 ===")

# 1. 物理红线：在你的 UC Volume 路径下，强行固化一个专属的检查点持久化目录
checkpoint_path = "/Volumes/dbacademy/streaming/vol_yuto_stream_source/_checkpoints/realtime_monitor/"

# 2. 🚀 释放微批次引擎：显式挂载 checkpointLocation 选项，一举击碎 SQLSTATE: 0A000！
query = df_alerts.writeStream \
    .format("memory") \
    .queryName("yuto_realtime_alerts_view") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start()

# 3. 等待引擎横扫 Volume 并平稳落盘
query.awaitTermination()

print("\n🎉 [⚡ 数据长征安全闭环] 引擎已成功在物理账本中卡好‘书签’，并平稳熄火释放集群！")

